# BIRD Retrieval Ablation — Qwen-32B + Query Expansion + BM25/Dense Retrieval

**Component 1** of the deterministic schema-linking ablation study. Layered on top of the [`baseline.ipynb`](baseline.ipynb) pure-LLM run by adding:

1. **Query expansion** via a *cheaper* LLM (`Qwen2.5-7B-Instruct`, BF16) — generates 3 paraphrases + extracted entities per BIRD question.
2. **Hybrid retrieval** (BM25 + dense embeddings via `BAAI/bge-small-en-v1.5`, fused with Reciprocal Rank Fusion) over a column-level corpus per database.
3. **Multi-query aggregation** — RRF across all expansion variants → top-N candidate tables.
4. **Linked-schema prompt** — Qwen-32B sees ONLY the retrieved candidate tables (with their columns), not the full schema dump.

Pipeline:
```
question + evidence
      ↓
[Qwen-7B] expand → 3 paraphrases + entity list
      ↓
expansion set Q = {q_orig, q_p1, q_p2, q_p3, e_1, ..., e_n}
      ↓
[BM25 + dense + RRF] each q → ranked tables (top-K each)
      ↓
RRF aggregate across Q → top-N candidate tables T
      ↓
build linked-schema prompt (T only, columns + FKs among T)
      ↓
[Qwen-32B] → SQL
```

**Comparison frame:**

| Run | Schema visible to Qwen-32B | EX |
|---|---|---|
| Baseline | full DB schema dump | 54.2 |
| **+ Retrieval (this run)** | **only retrieved candidate tables** | **?** |
| + Graph (future) | retrieval ∪ minimal connecting subgraph | ? |
| + Rewriter (future, full pipeline) | same + AST repair | 64.4 (paper target) |

Predictions land in `/content/predict_dev_retrieval.json` (separate from baseline / pipeline files).

**Disk note:** uses `unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit` — the *same* NF4-quantized Qwen-32B as `baseline.ipynb`, but pre-quantized on disk (~18 GB) instead of BF16-then-quantize-at-load (~65 GB). Required for free Colab disk caps when running both Qwen-32B + Qwen-7B in one session.

## 1. Setup — clone repo, install deps

Adds `rank_bm25` and `sentence-transformers` on top of the baseline dep set.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/mayursk2000/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents.git"
REPO_DIR = pathlib.Path("/content/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "sqlglot", "huggingface_hub", "requests", "tqdm", "psutil",
    "transformers>=4.45", "accelerate>=0.34", "bitsandbytes>=0.43", "sentencepiece",
    "rank_bm25", "sentence-transformers",
])
print("Repo:", REPO_DIR)

## 2. Secrets — `HF_TOKEN` from Colab Secrets (optional for Qwen)

In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN not available:", e)
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("HF_TOKEN set:", bool(HF_TOKEN))

## 3. GPU check

Both the SQL generator (Qwen-32B 4-bit, ~18 GB) and expansion model (Qwen-7B BF16, ~14 GB) are pinned. Plus the BGE encoder (~150 MB) and per-db retrieval indexes (~50 MB peak). Total resident: ~33 GB. Comfortable on a 95 GB G4 / 80 GB A100.

In [ ]:
import torch

GEN_MODEL_NAME       = "unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit"  # pre-quantized 4-bit (~18GB disk vs ~65GB BF16); same NF4 weights as in-memory bnb quant
EXPANSION_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"         # cheap LLM for query expansion
ENCODER_NAME         = "BAAI/bge-small-en-v1.5"           # dense retriever encoder

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  |  VRAM: {vram_gb:.1f} GB")
    needed = 18 + 14 + 1
    if vram_gb < needed:
        print(f"WARNING: <{needed} GB VRAM. Need ~18 GB (Qwen-32B 4-bit) + ~14 GB (Qwen-7B BF16) + 1 GB (encoder).")
    # Disk: ~18 GB (Qwen-32B pre-4bit) + ~14 GB (Qwen-7B BF16) + ~3 GB (BIRD) ≈ 35 GB. OK on free Colab (~107 GB).
else:
    print("No GPU detected. Set Runtime → Change runtime type → GPU.")

print("Generator:", GEN_MODEL_NAME)
print("Expansion:", EXPANSION_MODEL_NAME)
print("Encoder:  ", ENCODER_NAME)

## 4. Download BIRD dev — same downloader as baseline

In [ ]:
import re, zipfile, json, urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown

BIRD_ROOT = pathlib.Path("/content/bird")
BIRD_ROOT.mkdir(exist_ok=True)

BIRD_DEV_URL = "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"
zip_path = BIRD_ROOT / "dev.zip"

def _looks_like_html(p: pathlib.Path) -> bool:
    if not p.exists() or p.stat().st_size < 1_000_000:
        try:
            head = p.read_bytes()[:512].lower()
            return b"<!doctype html" in head or b"<html" in head
        except Exception:
            return True
    return False

def _drive_id(url: str):
    for pat in (r"/file/d/([A-Za-z0-9_-]{20,})", r"[?&]id=([A-Za-z0-9_-]{20,})"):
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None

if zip_path.exists() and _looks_like_html(zip_path):
    zip_path.unlink()

if BIRD_DEV_URL and not zip_path.exists():
    drive_id = _drive_id(BIRD_DEV_URL)
    if drive_id:
        gdown.download(id=drive_id, output=str(zip_path), quiet=False)
    else:
        urllib.request.urlretrieve(BIRD_DEV_URL, zip_path)

assert zip_path.exists() and not _looks_like_html(zip_path), "BIRD dev download failed."

if not list(BIRD_ROOT.rglob("dev.json")):
    print("Extracting ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BIRD_ROOT)
    for inner in BIRD_ROOT.rglob("dev_databases.zip"):
        with zipfile.ZipFile(inner) as z:
            z.extractall(inner.parent)
        inner.unlink()  # ~1.5 GB freed
    zip_path.unlink()   # ~1.5 GB freed
    print("Removed BIRD zips after extraction (frees ~3 GB).")

candidates = [p.parent for p in BIRD_ROOT.rglob("dev.json")]
assert candidates, f"dev.json not found under {BIRD_ROOT}."
BIRD_DEV = candidates[0]
BIRD_DBS = next(BIRD_DEV.rglob("dev_databases"), BIRD_DEV / "dev_databases")
print("BIRD_DEV =", BIRD_DEV)
print("BIRD_DBS =", BIRD_DBS)
print("Questions:", len(json.loads((BIRD_DEV / "dev.json").read_text())))
print("Databases:", len(list(BIRD_DBS.iterdir())) if BIRD_DBS.exists() else "missing")

## 5. Schema introspection — per-table view (not full-dump)

Unlike the baseline, retrieval needs structured per-table access (table descriptions, per-column descriptions, FK list). Reuse the prototype's `Schema`/`Table`/`Column` dataclasses so we can plug into the existing `HybridRetriever` machinery.

In [ ]:
import sqlite3, csv
from text2sql_agent_prototype.prototype import Schema, Table, Column, ForeignKey

_BIRD_TABLES_JSON: "dict | None" = None

def _load_dev_tables() -> dict:
    global _BIRD_TABLES_JSON
    if _BIRD_TABLES_JSON is None:
        path = next(BIRD_DEV.rglob("dev_tables.json"), None)
        if path is None:
            print("WARN: dev_tables.json not found.")
            _BIRD_TABLES_JSON = {}
        else:
            blocks = json.loads(path.read_text())
            _BIRD_TABLES_JSON = {b["db_id"]: b for b in blocks}
    return _BIRD_TABLES_JSON

def _read_column_descriptions(db_id: str) -> "dict[str, dict[str, str]]":
    desc_dir = BIRD_DBS / db_id / "database_description"
    out: "dict[str, dict[str, str]]" = {}
    if not desc_dir.exists():
        return out
    for csv_path in desc_dir.glob("*.csv"):
        tbl = csv_path.stem.lower()
        col_descs: "dict[str, str]" = {}
        try:
            with open(csv_path, encoding="utf-8", errors="replace") as f:
                for row in csv.DictReader(f):
                    col = (row.get("original_column_name") or "").strip()
                    desc = (row.get("column_description") or "").strip()
                    val_desc = (row.get("value_description") or "").strip()
                    full = " ; ".join(filter(None, [desc, val_desc]))
                    if col:
                        col_descs[col.lower()] = full
        except Exception as exc:
            print(f"  warn: couldn't read {csv_path.name}: {exc}")
        out[tbl] = col_descs
    return out

def schema_from_bird(db_id: str) -> Schema:
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    assert db_path.exists(), f"missing {db_path}"
    conn = sqlite3.connect(str(db_path))
    try:
        cur = conn.cursor()
        table_names = [r[0] for r in cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        ).fetchall()]
        col_descs = _read_column_descriptions(db_id)
        bird_meta = _load_dev_tables().get(db_id)

        tables: "dict[str, Table]" = {}
        for tname in table_names:
            cols_info = cur.execute(f'PRAGMA table_info("{tname}")').fetchall()
            tbl_descs = col_descs.get(tname.lower(), {})
            columns = tuple(
                Column(
                    name=c[1],
                    description=" | ".join(filter(None, [
                        c[2].lower() if c[2] else "",
                        tbl_descs.get(c[1].lower(), ""),
                    ])),
                )
                for c in cols_info
            )
            tbl_desc = tname.replace("_", " ")
            if bird_meta and tname in bird_meta.get("table_names_original", []):
                idx = bird_meta["table_names_original"].index(tname)
                if idx < len(bird_meta.get("table_names", [])):
                    tbl_desc = bird_meta["table_names"][idx] or tbl_desc
            tables[tname] = Table(name=tname, columns=columns, description=tbl_desc)

        foreign_keys: "list[ForeignKey]" = []
        if bird_meta:
            names_orig = bird_meta["table_names_original"]
            cols_orig  = bird_meta["column_names_original"]
            for fk_pair in bird_meta.get("foreign_keys", []):
                l_idx, r_idx = fk_pair[0], fk_pair[1]
                l_tbl = names_orig[cols_orig[l_idx][0]]; l_col = cols_orig[l_idx][1]
                r_tbl = names_orig[cols_orig[r_idx][0]]; r_col = cols_orig[r_idx][1]
                if l_tbl in tables and r_tbl in tables:
                    foreign_keys.append(ForeignKey(
                        left_table=l_tbl, left_column=l_col,
                        right_table=r_tbl, right_column=r_col,
                    ))
        seen = {(fk.left_table, fk.left_column, fk.right_table, fk.right_column)
                for fk in foreign_keys}
        for tname in table_names:
            for fk in cur.execute(f'PRAGMA foreign_key_list("{tname}")').fetchall():
                _, _, ref_table, from_col, to_col, *_ = fk
                if ref_table in tables and (tname, from_col, ref_table, to_col) not in seen:
                    foreign_keys.append(ForeignKey(
                        left_table=tname, left_column=from_col,
                        right_table=ref_table, right_column=to_col,
                    ))

        return Schema(tables=tables, foreign_keys=foreign_keys)
    finally:
        conn.close()

def load_dev(bird_dev_dir: pathlib.Path) -> list:
    return json.loads((bird_dev_dir / "dev.json").read_text())

## 6. Sanity check

In [ ]:
print("BIRD_DEV =", BIRD_DEV)
print("BIRD_DBS =", BIRD_DBS)
questions = load_dev(BIRD_DEV)
print(f"Total questions: {len(questions)}")
sch = schema_from_bird(questions[0]["db_id"])
print(f"\nSample schema: db={questions[0]['db_id']}  tables={len(sch.tables)}  fks={len(sch.foreign_keys)}")
print("Tables:", ", ".join(sch.tables.keys()))

## 7. Hybrid retriever — BM25 + BGE dense embeddings + RRF (column-level corpus)

Reused from the pipeline notebook. One encoder, one BM25 index per `db_id`, cached so we don't re-embed for every question. RRF over BM25 + dense rankings, then table-score = max RRF over the table's columns.

In [ ]:
import re as _re
from collections import defaultdict
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from text2sql_agent_prototype.prototype import (
    HybridRetriever, RetrievalResult, RetrievalMatch,
)

print(f"Loading encoder {ENCODER_NAME} ...")
encoder = SentenceTransformer(
    ENCODER_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
_dim = (encoder.get_embedding_dimension()
        if hasattr(encoder, "get_embedding_dimension")
        else encoder.get_sentence_embedding_dimension())
print(f"Encoder loaded ({_dim}-dim).")


def _smart_split(text: str) -> "list[str]":
    """Tokenize for BM25: split on non-alpha, then CamelCase, lowercase, drop short tokens."""
    out: "list[str]" = []
    for word in _re.findall(r"[A-Za-z0-9]+", text):
        for piece in _re.findall(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|[0-9]+", word):
            t = piece.lower()
            if len(t) >= 2:
                out.append(t)
    return out


class BIRDHybridRetriever(HybridRetriever):
    """BM25 + dense, fused via RRF. Column-level corpus."""

    RRF_K = 60

    def __init__(self, schema: Schema, db_id: str, top_k: int = 8):
        self.schema = schema
        self.db_id = db_id
        self.top_k = top_k
        self.min_score = 0.0

        self.docs: "list[str]" = []
        self.doc_tables: "list[str]" = []
        for tname, table in schema.tables.items():
            for c in table.columns:
                doc = (
                    f"Table {tname} ({table.description}). "
                    f"Column {c.name}: {c.description or ''}"
                )
                self.docs.append(doc)
                self.doc_tables.append(tname)

        self.bm25 = BM25Okapi([_smart_split(d) for d in self.docs])
        self.dense = encoder.encode(
            self.docs,
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        )

    def _rank_tables(self, query: str) -> "list[tuple[str, float]]":
        """Return [(table_name, rrf_score), ...] sorted descending. NOT capped at top_k."""
        if not self.docs:
            return []
        bm25_scores = self.bm25.get_scores(_smart_split(query))
        bm25_rank = {int(i): r for r, i in enumerate(np.argsort(-bm25_scores))}
        q_emb = encoder.encode(
            [query], normalize_embeddings=True,
            show_progress_bar=False, convert_to_numpy=True,
        )[0]
        dense_scores = self.dense @ q_emb
        dense_rank = {int(i): r for r, i in enumerate(np.argsort(-dense_scores))}
        K = self.RRF_K
        rrf = {i: 1.0 / (K + bm25_rank[i]) + 1.0 / (K + dense_rank[i])
               for i in range(len(self.docs))}
        table_score: "dict[str, float]" = {}
        for i, tname in enumerate(self.doc_tables):
            s = rrf[i]
            if tname not in table_score or s > table_score[tname]:
                table_score[tname] = s
        return sorted(table_score.items(), key=lambda kv: -kv[1])

    def retrieve(self, query: str) -> RetrievalResult:
        ranked = self._rank_tables(query)[: self.top_k]
        matches = [
            RetrievalMatch(
                table=t, lexical_score=0.0, semantic_score=0.0,
                score=float(s), matched_terms=[],
            ) for t, s in ranked
        ]
        return RetrievalResult(
            query=query,
            candidate_tables=[m.table for m in matches],
            matches=matches,
        )


_RETRIEVER_CACHE: "dict[str, BIRDHybridRetriever]" = {}

def get_retriever(db_id: str) -> BIRDHybridRetriever:
    if db_id not in _RETRIEVER_CACHE:
        sch = schema_from_bird(db_id)
        _RETRIEVER_CACHE[db_id] = BIRDHybridRetriever(sch, db_id)
    return _RETRIEVER_CACHE[db_id]


# ---- Probe: retriever on Q0 ----
_probe_q   = questions[0]
_probe_retr = get_retriever(_probe_q["db_id"])
_probe_text = _probe_q["question"] + " " + (_probe_q.get("evidence") or "")
_probe_result = _probe_retr.retrieve(_probe_text)
print(f"\nProbe q (db={_probe_q['db_id']}): {_probe_text[:120]}")
print(f"Top-{len(_probe_result.candidate_tables)} tables → "
      f"{', '.join(_probe_result.candidate_tables)}")

## 8. Load expansion LLM — Qwen2.5-7B-Instruct (BF16)

Loaded *before* Qwen-32B because BF16 model load is faster. Kept resident throughout the run — expansion runs once per question, ~50 tokens out, ~0.3–0.5 sec each.

**Note:** non-Coder variant. Paraphrasing is a general-LM task; the Coder variant biases toward emitting SQL-shaped strings, the opposite of what we want here.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import re, gc, json as _json

print(f"Loading expansion model {EXPANSION_MODEL_NAME} (BF16) ...")
exp_tokenizer = AutoTokenizer.from_pretrained(EXPANSION_MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
exp_tokenizer.padding_side = "left"
exp_tokenizer.truncation_side = "left"
if exp_tokenizer.pad_token_id is None:
    exp_tokenizer.pad_token = exp_tokenizer.eos_token
    exp_tokenizer.pad_token_id = exp_tokenizer.eos_token_id

exp_model = AutoModelForCausalLM.from_pretrained(
    EXPANSION_MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
exp_model.eval()
print("Expansion model loaded.")

EXPANSION_SYSTEM = (
    "You are a query-expansion assistant for a Text-to-SQL retrieval system. "
    "You receive a natural-language database question and optional evidence "
    "hints. Your job: emit JSON with two fields:\n"
    '  "paraphrases": 3 alternate phrasings of the question, each preserving '
    "intent but using different vocabulary. Aim for synonyms a database schema "
    "might use (e.g. 'students' -> 'enrollment', 'school' -> 'institution').\n"
    '  "entities": short noun phrases pulled from the question/evidence that '
    "look like database tables, columns, or filter values. Include backticked "
    "tokens from the evidence verbatim (without backticks). 1-8 entities.\n\n"
    "Output ONLY the JSON object. No prose, no markdown fences."
)

EXPANSION_FEWSHOT = [
    {"role": "user", "content":
        "Question: How many charter schools in Fresno County have free meal "
        "counts above 100?\n"
        "Evidence: Charter schools refers to `Charter School (Y/N)` = 1 in the frpm"},
    {"role": "assistant", "content":
        '{"paraphrases": ["Count charter schools located in Fresno County with '
        'free meal counts greater than 100", "How many institutions categorized '
        'as charter in Fresno County have more than 100 free meals", "Number '
        'of charter-status schools in the Fresno County area where free meal '
        'count exceeds 100"], "entities": ["charter schools", "Fresno County", '
        '"free meal count", "Charter School (Y/N)", "frpm"]}'},
]

_JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def _parse_expansion(raw: str, fallback_question: str) -> "dict":
    """Parse expansion output. Return dict with 'paraphrases' and 'entities'.
    On any failure, fall back to original question with no entities."""
    m = _JSON_RE.search(raw)
    if not m:
        return {"paraphrases": [fallback_question], "entities": []}
    try:
        obj = _json.loads(m.group(0))
    except Exception:
        return {"paraphrases": [fallback_question], "entities": []}
    paras = obj.get("paraphrases") or []
    if not isinstance(paras, list):
        paras = [str(paras)]
    paras = [str(p).strip() for p in paras if str(p).strip()][:3]
    if not paras:
        paras = [fallback_question]
    ents  = obj.get("entities") or []
    if not isinstance(ents, list):
        ents = [str(ents)]
    ents = [str(e).strip() for e in ents if str(e).strip()][:8]
    return {"paraphrases": paras, "entities": ents}


def expand_query(question: str, evidence: str = "") -> "dict":
    """Run Qwen-7B once. Return {'paraphrases': [...], 'entities': [...]}."""
    user = f"Question: {question}"
    if evidence:
        user += f"\nEvidence: {evidence}"
    msgs = [
        {"role": "system", "content": EXPANSION_SYSTEM},
        *EXPANSION_FEWSHOT,
        {"role": "user", "content": user},
    ]
    prompt = exp_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = exp_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(exp_model.device)
    with torch.inference_mode():
        out = exp_model.generate(
            **inputs, max_new_tokens=256, do_sample=False,
            pad_token_id=exp_tokenizer.pad_token_id,
        )
    raw = exp_tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    del out, inputs
    try: torch.cuda.empty_cache()
    except Exception: pass
    return _parse_expansion(raw, question)


def expand_query_batch(items: "list[tuple[str, str]]", batch_size: int = 16) -> "list[dict]":
    """items: list of (question, evidence). Returns list of expansion dicts."""
    if not items:
        return []
    results: "list[dict]" = []
    for start in range(0, len(items), batch_size):
        chunk = items[start:start + batch_size]
        prompts = []
        for q, ev in chunk:
            user = f"Question: {q}"
            if ev:
                user += f"\nEvidence: {ev}"
            msgs = [
                {"role": "system", "content": EXPANSION_SYSTEM},
                *EXPANSION_FEWSHOT,
                {"role": "user", "content": user},
            ]
            prompts.append(exp_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        try:
            inputs = exp_tokenizer(prompts, return_tensors="pt", padding=True,
                                    truncation=True, max_length=2048).to(exp_model.device)
            with torch.inference_mode():
                out = exp_model.generate(
                    **inputs, max_new_tokens=256, do_sample=False,
                    pad_token_id=exp_tokenizer.pad_token_id,
                )
            decoded = exp_tokenizer.batch_decode(
                out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
            )
            del out, inputs
            try: torch.cuda.empty_cache()
            except Exception: pass
        except BaseException as exc:
            if isinstance(exc, (KeyboardInterrupt, SystemExit)):
                raise
            print(f"  expansion batch failed ({type(exc).__name__}: {str(exc)[:80]})")
            decoded = [""] * len(chunk)
        for raw, (q, _ev) in zip(decoded, chunk):
            results.append(_parse_expansion(raw, q))
    return results

print("expand_query / expand_query_batch ready.")

## 9. Smoke test — query expansion on Q0

In [ ]:
q0 = questions[0]
print("Question: ", q0["question"])
print("Evidence: ", q0.get("evidence", ""))
print("-" * 80)
exp = expand_query(q0["question"], q0.get("evidence", "") or "")
print("Paraphrases:")
for i, p in enumerate(exp["paraphrases"], 1):
    print(f"  [{i}] {p}")
print("Entities:", exp["entities"])

## 10. Multi-query retrieval — RRF across expansion variants

For each expansion variant we get a per-table RRF score. We RRF-aggregate *those rankings* to get the final candidate set. Belt-and-braces: also include the original question as one of the variants so a bad expansion doesn't tank retrieval.

In [ ]:
N_CANDIDATE_TABLES = 5  # how many tables Qwen-32B will see

def retrieve_with_expansion(
    db_id: str, question: str, evidence: str, expansion: "dict",
    n_tables: int = N_CANDIDATE_TABLES,
) -> "tuple[list[str], dict]":
    """Run the retriever once per expansion variant, RRF-aggregate the per-variant
    table rankings, return the top-N candidate tables and a debug dict.

    Variants:
      - the original question + evidence (anchor)
      - each paraphrase (3)
      - each entity (1-8)

    All variants are weighted equally in RRF. The original is included as one
    variant so a bad expansion doesn't fully replace the signal.
    """
    retriever = get_retriever(db_id)

    variants: "list[str]" = []
    anchor = (question + " " + evidence).strip() if evidence else question
    variants.append(anchor)
    variants.extend(expansion.get("paraphrases", []))
    variants.extend(expansion.get("entities", []))

    K = 60  # standard RRF
    table_rrf: "dict[str, float]" = defaultdict(float)
    per_variant_top: "list[list[str]]" = []
    for v in variants:
        if not v.strip():
            continue
        ranked = retriever._rank_tables(v)
        per_variant_top.append([t for t, _ in ranked[:n_tables]])
        for rank, (tname, _score) in enumerate(ranked):
            table_rrf[tname] += 1.0 / (K + rank)

    final_ranked = sorted(table_rrf.items(), key=lambda kv: -kv[1])[:n_tables]
    candidate_tables = [t for t, _ in final_ranked]

    return candidate_tables, {
        "n_variants": len(variants),
        "per_variant_top": per_variant_top,
        "final_ranked": final_ranked,
    }


# Smoke
exp_q0 = expand_query(q0["question"], q0.get("evidence", "") or "")
cands, debug = retrieve_with_expansion(q0["db_id"], q0["question"], q0.get("evidence", "") or "", exp_q0)
print(f"db={q0['db_id']}  n_variants={debug['n_variants']}")
print(f"Final candidate tables (top-{N_CANDIDATE_TABLES}): {cands}")
print(f"\nPer-variant top-{N_CANDIDATE_TABLES}:")
for i, v_top in enumerate(debug["per_variant_top"]):
    print(f"  variant {i}: {v_top}")

## 11. Load Qwen-32B (4-bit) and define linked-schema generator

Same model and same prompt rules as `baseline.ipynb`, but the schema dump is replaced with **only the candidate tables** (with column descriptions surfaced for each), plus the FK list filtered to the candidate set.

In [ ]:
from transformers import BitsAndBytesConfig
import re, gc

USE_4BIT          = True
MAX_NEW_TOKENS    = 256
MAX_INPUT_TOKENS  = 8192
MAX_DESC_LEN      = 100
MAX_TABLE_COLUMNS = 80

print(f"Loading {GEN_MODEL_NAME} (4-bit={USE_4BIT}) ...")
# Note: GEN_MODEL_NAME is already 4-bit on disk (unsloth/...-bnb-4bit). Passing
# BitsAndBytesConfig is harmless — bnb just inherits the embedded quant config.

_bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
gen_tokenizer.padding_side = "left"
gen_tokenizer.truncation_side = "left"
if gen_tokenizer.pad_token_id is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token
    gen_tokenizer.pad_token_id = gen_tokenizer.eos_token_id

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=_bnb_cfg,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
gen_model.eval()
print("Generator loaded.")

_FENCE = re.compile(r"^```(?:sql|sqlite)?\s*|\s*```$", re.IGNORECASE | re.MULTILINE)

GEN_SYSTEM_PROMPT = (
    "You are an expert Text-to-SQL assistant for SQLite, evaluated on BIRD. "
    "You receive: (1) a SCHEMA-LINKED subset of the database (only the tables "
    "deemed relevant by retrieval), (2) optional BIRD evidence with column "
    "hints, (3) a natural-language question.\n\n"
    "STRICT RULES:\n"
    "- If the evidence names a specific column in backticks (e.g. "
    "`Free Meal Count (K-12)`), USE THAT EXACT COLUMN. Do not substitute a "
    "similar-sounding column even if it seems to mean the same thing.\n"
    "- If the evidence provides a formula (e.g. \"rate = A / B\"), COMPUTE "
    "that expression literally. When the formula is a ratio, CAST the numerator "
    "to REAL (e.g. CAST(A AS REAL) / B) so SQLite doesn't integer-divide.\n"
    "- If the evidence gives a literal value (e.g. \"= 1\" or \"= 'Directly funded'\"), "
    "use that EXACT value, including punctuation, capitalization, and spacing.\n"
    "- Use ONLY tables and columns that appear in the linked schema below.\n"
    "- Quote column names containing spaces or special characters with double quotes.\n"
    "- Return EXACTLY ONE SQLite SELECT statement. No prose, no code fences."
)

def _trim(s: str) -> str:
    s = (s or "").replace("\r", " ").replace("\n", " ").replace("\t", " ").strip()
    if len(s) > MAX_DESC_LEN:
        s = s[:MAX_DESC_LEN].rstrip() + "…"
    return s

def linked_schema_prompt(db_id: str, candidate_tables: "list[str]") -> str:
    """Build a CREATE-TABLE-style prompt for ONLY the candidate tables, plus
    the FK list filtered to those tables."""
    schema = schema_from_bird(db_id)
    blocks: "list[str]" = []
    for tname in candidate_tables:
        if tname not in schema.tables:
            continue
        table = schema.tables[tname]
        lines = []
        for c in table.columns[:MAX_TABLE_COLUMNS]:
            desc = _trim(c.description or "")
            base = f'    "{c.name}"'
            lines.append(f"{base}  -- {desc}" if desc else base)
        if len(table.columns) > MAX_TABLE_COLUMNS:
            lines.append(f"    -- ... {len(table.columns) - MAX_TABLE_COLUMNS} more columns omitted")
        desc_line = f' -- {table.description}' if table.description else ""
        blocks.append(f'CREATE TABLE "{tname}" ({desc_line}\n' + ",\n".join(lines) + "\n);")

    cand_set = set(candidate_tables)
    fk_lines = [
        f'  "{fk.left_table}"."{fk.left_column}" -> "{fk.right_table}"."{fk.right_column}"'
        for fk in schema.foreign_keys
        if fk.left_table in cand_set and fk.right_table in cand_set
    ]
    out = "\n\n".join(blocks)
    if fk_lines:
        out += "\n\n-- Foreign keys (within retrieved tables):\n" + "\n".join(fk_lines)
    return out

def _build_gen_messages(db_id: str, question: str, evidence: str, candidate_tables: "list[str]") -> list:
    user_parts = [
        f"Linked schema (top-{len(candidate_tables)} tables retrieved by BM25+dense+RRF):\n"
        f"{linked_schema_prompt(db_id, candidate_tables)}"
    ]
    if evidence:
        user_parts.append(
            "BIRD evidence — REQUIRED column hints. If a column or value here is "
            f"given in backticks, use it EXACTLY as written:\n{evidence}"
        )
    user_parts.append(f"Question:\n{question}")
    user_parts.append("Output only the SELECT statement.")
    return [
        {"role": "system", "content": GEN_SYSTEM_PROMPT},
        {"role": "user",   "content": "\n\n".join(user_parts)},
    ]

def _clean_sql(text: str) -> str:
    text = _FENCE.sub("", text).strip()
    m = re.search(r"\b(SELECT|WITH)\b", text, re.IGNORECASE)
    if m:
        text = text[m.start():]
    return text.strip().rstrip(";").strip()

def _validate_sql(sql: str) -> str:
    if not re.match(r"(?i)^\s*(select|with)\b", sql):
        raise ValueError(f"Non-SELECT output: {sql[:120]}")
    return sql

def _free_cache():
    try: torch.cuda.empty_cache()
    except Exception: pass

def generate_sql_one(db_id: str, question: str, evidence: str, candidate_tables: "list[str]") -> str:
    msgs = _build_gen_messages(db_id, question, evidence, candidate_tables)
    prompt = gen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(gen_model.device)
    with torch.inference_mode():
        out = gen_model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=gen_tokenizer.pad_token_id,
        )
    new_tokens = out[0, inputs.input_ids.shape[1]:]
    raw = gen_tokenizer.decode(new_tokens, skip_special_tokens=True)
    del out, inputs, new_tokens
    _free_cache()
    return _validate_sql(_clean_sql(raw))

def generate_sql_batch(items: "list[tuple[str, str, str, list[str]]]",
                       max_input_length: int = MAX_INPUT_TOKENS) -> "list[str]":
    """items: list of (db_id, question, evidence, candidate_tables)."""
    if not items:
        return []
    out = inputs = new_token_rows = None
    try:
        prompts = [
            gen_tokenizer.apply_chat_template(
                _build_gen_messages(db, q, ev, cands),
                tokenize=False, add_generation_prompt=True,
            )
            for (db, q, ev, cands) in items
        ]
        prompt_lengths = [len(gen_tokenizer(p, add_special_tokens=False).input_ids) for p in prompts]
        if prompt_lengths and max(prompt_lengths) > max_input_length:
            print(f"  truncating gen batch: max_tokens={max(prompt_lengths)} cap={max_input_length}")
        inputs = gen_tokenizer(prompts, return_tensors="pt", padding=True,
                               truncation=True, max_length=max_input_length).to(gen_model.device)
        with torch.inference_mode():
            out = gen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     pad_token_id=gen_tokenizer.pad_token_id)
        input_len = inputs.input_ids.shape[1]
        new_token_rows = out[:, input_len:]
        decoded = gen_tokenizer.batch_decode(new_token_rows, skip_special_tokens=True)
    except BaseException as exc:
        if isinstance(exc, (KeyboardInterrupt, SystemExit)):
            raise
        print(f"  gen batch failed ({type(exc).__name__}: {str(exc)[:120]})  batch={len(items)}")
        del out, inputs
        _free_cache(); gc.collect()
        if len(items) > 1:
            mid = len(items) // 2
            return generate_sql_batch(items[:mid], max_input_length) + generate_sql_batch(items[mid:], max_input_length)
        try:
            return [generate_sql_one(*items[0])]
        except Exception:
            return ["SELECT 1"]
    finally:
        try: del out, inputs, new_token_rows
        except NameError: pass
        _free_cache()
    return [
        (lambda raw: _validate_sql(_clean_sql(raw)) if re.search(r"\b(SELECT|WITH)\b", raw, re.IGNORECASE) else "SELECT 1")(r)
        if r else "SELECT 1" for r in decoded
    ]

print("SQL generator ready.")

## 12. Smoke test — full retrieval + generation on Q0

In [ ]:
cands_q0, _debug = retrieve_with_expansion(
    q0["db_id"], q0["question"], q0.get("evidence", "") or "", exp_q0,
)
sql = generate_sql_one(q0["db_id"], q0["question"], q0.get("evidence", "") or "", cands_q0)
print(f"db_id:    {q0['db_id']}")
print(f"question: {q0['question']}")
print(f"evidence: {q0.get('evidence', '')}")
print(f"candidate_tables: {cands_q0}")
print(f"\nGenerated SQL: {sql}")
print(f"Gold SQL:      {q0['SQL']}")

## 13. Run on full BIRD dev → `predict_dev_retrieval.json`

Per-question pipeline:
1. Expansion via Qwen-7B (batched per db)
2. Multi-query RRF retrieval over the per-db column corpus
3. Linked-schema prompt build
4. Qwen-32B generation (batched)

Predictions checkpointed every 50 questions. Retrieval debug log written to `/content/retrieval_log.jsonl` for post-hoc analysis.

In [ ]:
from tqdm.auto import tqdm
from collections import defaultdict
import gc, psutil, os, torch, json

N            = len(questions)
GEN_BATCH    = int(os.environ.get("RET_GEN_BATCH", "8"))
EXP_BATCH    = int(os.environ.get("RET_EXP_BATCH", "16"))
PRED_PATH    = pathlib.Path("/content/predict_dev_retrieval.json")
LOG_PATH     = pathlib.Path("/content/retrieval_log.jsonl")

predictions: "dict[str, str]" = {}
if PRED_PATH.exists():
    predictions = json.loads(PRED_PATH.read_text())
    print(f"Resuming with {len(predictions)} predictions on disk")
if not predictions and LOG_PATH.exists():
    LOG_PATH.unlink()

def _mem_str() -> str:
    rss = psutil.Process(os.getpid()).memory_info().rss / 1e9
    avail = psutil.virtual_memory().available / 1e9
    gpu = ""
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        gpu = f"  GPU free={free/1e9:.1f}GB"
    return f"RSS={rss:.1f}GB  free={avail:.1f}GB{gpu}"

def write_predictions(path, preds):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(preds, f)

def append_log(path, lines):
    if not lines:
        return
    with open(path, "a", encoding="utf-8") as f:
        for r in lines:
            f.write(json.dumps(r) + "\n")

groups: "dict[str, list]" = defaultdict(list)
for i, q in enumerate(questions[:N]):
    groups[q["db_id"]].append((i, q))

print(f"BIRD dev: {N} questions across {len(groups)} databases")
print(f"Memory at start: {_mem_str()}")

total_done = len(predictions)
pbar = tqdm(total=N, initial=total_done, desc="BIRD dev (retrieval)")
last_checkpoint = total_done

for db_id, qlist in groups.items():
    qlist_remaining = [(i, q) for (i, q) in qlist if str(q.get("question_id", i)) not in predictions]
    if not qlist_remaining:
        continue

    # Warm retriever (builds BM25 + dense indexes for this db once).
    retr = get_retriever(db_id)
    print(f"  [{db_id}] {len(qlist_remaining)} questions, "
          f"corpus={len(retr.docs)} columns  {_mem_str()}")

    # ---- Pass 1: expand all remaining questions for this db (batched) ----
    exp_inputs = [(q["question"], q.get("evidence", "") or "") for (_i, q) in qlist_remaining]
    expansions = expand_query_batch(exp_inputs, batch_size=EXP_BATCH)

    # ---- Pass 2: retrieve candidates per question ----
    candidate_table_lists: "list[list[str]]" = []
    log_buf: "list[dict]" = []
    for (orig_i, q), exp in zip(qlist_remaining, expansions):
        qid = str(q.get("question_id", orig_i))
        cands, debug = retrieve_with_expansion(
            db_id, q["question"], q.get("evidence", "") or "", exp,
        )
        candidate_table_lists.append(cands)
        log_buf.append({
            "qid": qid, "db_id": db_id,
            "paraphrases": exp.get("paraphrases", []),
            "entities":    exp.get("entities", []),
            "candidate_tables": cands,
            "n_variants": debug["n_variants"],
        })
    append_log(LOG_PATH, log_buf)

    # ---- Pass 3: SQL generation in batches ----
    pending = list(zip(qlist_remaining, candidate_table_lists))
    for start in range(0, len(pending), GEN_BATCH):
        chunk = pending[start:start + GEN_BATCH]
        items = [
            (db_id, q["question"], q.get("evidence", "") or "", cands)
            for ((_i, q), cands) in chunk
        ]
        sqls = generate_sql_batch(items)
        for ((orig_i, q), _cands), sql in zip(chunk, sqls):
            qid = str(q.get("question_id", orig_i))
            sql = (sql or "SELECT 1").replace("\n", " ").strip()
            predictions[qid] = f"{sql}\t----- bird -----\t{db_id}"
            pbar.update(1)
        gc.collect()
        try: torch.cuda.empty_cache()
        except Exception: pass
        if len(predictions) - last_checkpoint >= 50:
            write_predictions(PRED_PATH, predictions)
            last_checkpoint = len(predictions)

    write_predictions(PRED_PATH, predictions)
    print(f"  [{db_id}] done. {_mem_str()}")

pbar.close()
write_predictions(PRED_PATH, predictions)
print(f"\nWrote {PRED_PATH} ({len(predictions)} predictions)")
print(f"Wrote {LOG_PATH} (retrieval debug)")
print(f"Memory at end: {_mem_str()}")

## 14. Local metrics — Join Acc + Exec Acc + Retrieval Recall

Same EX/Join-F1 as `baseline.ipynb`, plus a **retrieval-table-recall** metric: did the retrieved candidate set include every table the gold SQL actually uses? This is the upstream signal — if recall is low, retrieval is the bottleneck (not the LLM).

In [ ]:
import sqlite3, json, time
from collections import defaultdict
import sqlglot
from sqlglot import exp as _exp

PRED_PATH = pathlib.Path("/content/predict_dev_retrieval.json")
LOG_PATH  = pathlib.Path("/content/retrieval_log.jsonl")
predictions = json.loads(PRED_PATH.read_text())
qmap = {str(q.get("question_id", i)): q for i, q in enumerate(questions)}
ret_log = {}
if LOG_PATH.exists():
    for line in LOG_PATH.read_text().splitlines():
        if line.strip():
            try:
                obj = json.loads(line); ret_log[obj["qid"]] = obj
            except Exception:
                pass

TIMEOUT_S = 10
SQLITE_PROGRESS_STEPS = 10_000
_NULL = "\x00NULL\x00"

def run_sql(db_id: str, sql: str):
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(str(db_path), timeout=TIMEOUT_S)
    try:
        conn.execute(f"PRAGMA busy_timeout = {TIMEOUT_S * 1000}")
        deadline = time.monotonic() + TIMEOUT_S
        conn.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, SQLITE_PROGRESS_STEPS)
        return conn.execute(sql).fetchall(), None
    except Exception as exc:
        return None, str(exc)
    finally:
        try: conn.set_progress_handler(None, 0)
        except Exception: pass
        conn.close()

def normalize(rows):
    if rows is None:
        return None
    return sorted(tuple(_NULL if c is None else str(c) for c in r) for r in rows)

def extract_joins(sql: str):
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception:
        return set()
    alias_map = {}
    for t in tree.find_all(_exp.Table):
        name = (t.name or "").lower()
        alias = (t.alias or t.name or "").lower()
        if name and alias:
            alias_map[alias] = name
    joins = set()
    for eq in tree.find_all(_exp.EQ):
        left, right = eq.this, eq.expression
        if isinstance(left, _exp.Column) and isinstance(right, _exp.Column):
            lt = (left.table or "").lower(); rt = (right.table or "").lower()
            if lt and rt and lt in alias_map and rt in alias_map and lt != rt:
                a = f"{alias_map[lt]}.{left.name.lower()}"
                b = f"{alias_map[rt]}.{right.name.lower()}"
                joins.add(frozenset([a, b]))
    return joins

def join_f1(pred_sql, gold_sql):
    gold = extract_joins(gold_sql)
    if not gold:
        return None
    pred = extract_joins(pred_sql)
    if not pred:
        return 0.0
    tp = len(pred & gold)
    if tp == 0:
        return 0.0
    p = tp / len(pred); r = tp / len(gold)
    return 2 * p * r / (p + r)

def gold_tables(sql: str):
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception:
        return set()
    return {(t.name or "").lower() for t in tree.find_all(_exp.Table) if t.name}

results = []
for i, q in enumerate(questions):
    qid = str(q.get("question_id", i))
    line = predictions.get(qid)
    if line is None:
        pred_sql = "SELECT 1"; missing = True
    else:
        pred_sql = line.split("\t----- bird -----\t", 1)[0].strip(); missing = False
    is_fallback = pred_sql.rstrip(";").strip().upper() == "SELECT 1"

    pred_rows, pred_err = run_sql(q["db_id"], pred_sql)
    gold_rows, gold_err = run_sql(q["db_id"], q["SQL"])

    if missing: ex = "missing_prediction"
    elif pred_err:      ex = "exec_error"
    elif gold_err:      ex = "gold_error"
    elif normalize(pred_rows) == normalize(gold_rows): ex = "correct"
    else:               ex = "wrong_result"

    log_info = ret_log.get(qid, {})
    cand_set  = {t.lower() for t in log_info.get("candidate_tables", [])}
    gold_set  = gold_tables(q["SQL"])
    ret_recall = (len(cand_set & gold_set) / len(gold_set)) if gold_set else None

    results.append({
        "qid": qid, "db_id": q["db_id"], "difficulty": q.get("difficulty", "?"),
        "ex": ex, "pred_err": pred_err, "fallback": is_fallback, "missing": missing,
        "jf1": join_f1(pred_sql, q["SQL"]), "pred_sql": pred_sql,
        "ret_recall": ret_recall, "candidate_tables": list(cand_set), "gold_tables": list(gold_set),
        "log": log_info,
    })

total = len(results)
ex_correct = sum(r["ex"] == "correct" for r in results)
ex_acc = ex_correct / max(total, 1) * 100
scored_join = [r for r in results if r["jf1"] is not None]
join_acc = (sum(r["jf1"] for r in scored_join) / max(len(scored_join), 1)) * 100
scored_recall = [r for r in results if r["ret_recall"] is not None]
ret_recall_avg = (sum(r["ret_recall"] for r in scored_recall) / max(len(scored_recall), 1)) * 100

print(f"=== Retrieval ablation (Qwen-32B + Qwen-7B expansion + BM25/dense+RRF) — {total} predictions ===\n")
print(f"{'Method':<32s}{'Join Acc.':>12s}{'Exec. Acc.':>13s}")
print(f"{'-'*57}")
print(f"{'Prior Work [BIRD GPT-4]':<32s}{58.2:>12.1f}{54.9:>13.1f}")
print(f"{'Baseline (Qwen-32B, full)':<32s}{66.3:>12.1f}{54.2:>13.1f}")
print(f"{'+ Retrieval (this run)':<32s}{join_acc:>12.1f}{ex_acc:>13.1f}")
print(f"{'Ours target (full pipeline)':<32s}{73.8:>12.1f}{64.4:>13.1f}")

print(f"\nUpstream signal:")
print(f"  Retrieval-table recall: {ret_recall_avg:.1f}%   (top-{N_CANDIDATE_TABLES} included every gold table?)")
print(f"  Join Acc. averaged over {len(scored_join)} multi-table questions; "
      f"{total - len(scored_join)} single-table excluded.")

print(f"\nEX status:  correct={ex_correct}  "
      f"wrong={sum(r['ex']=='wrong_result' for r in results)}  "
      f"err={sum(r['ex']=='exec_error' for r in results)}  "
      f"missing={sum(r['ex']=='missing_prediction' for r in results)}  "
      f"fallback={sum(r['fallback'] for r in results)}")

print(f"\n--- By difficulty ---")
print(f"{'difficulty':<14s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'RetRecall':>11s}{'cases':>8s}")
for d in ("simple", "moderate", "challenging", "?"):
    bucket = [r for r in results if r["difficulty"] == d]
    if not bucket:
        continue
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    br = [r for r in bucket if r["ret_recall"] is not None]
    rr = (sum(r["ret_recall"] for r in br) / max(len(br), 1)) * 100 if br else float("nan")
    print(f"  {d:<12s}{j:>11.1f}{e:>12.1f}{rr:>11.1f}{len(bucket):>8d}")

print(f"\n--- By db_id ---")
print(f"{'db_id':<26s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'RetRecall':>11s}{'cases':>8s}")
by_db = defaultdict(list)
for r in results:
    by_db[r["db_id"]].append(r)
for d, bucket in sorted(by_db.items()):
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    br = [r for r in bucket if r["ret_recall"] is not None]
    rr = (sum(r["ret_recall"] for r in br) / max(len(br), 1)) * 100 if br else float("nan")
    print(f"  {d:<24s}{j:>11.1f}{e:>12.1f}{rr:>11.1f}{len(bucket):>8d}")

print("\n--- First 5 EX failures (with retrieval diagnosis) ---")
shown = 0
for r in results:
    if r["ex"] == "correct" or shown >= 5:
        continue
    q = qmap[r["qid"]]
    jf = "—" if r["jf1"] is None else f"{r['jf1']:.2f}"
    rr = "—" if r["ret_recall"] is None else f"{r['ret_recall']*100:.0f}%"
    missing_tables = set(r["gold_tables"]) - set(r["candidate_tables"])
    print(f"\n  qid={r['qid']} ({r['difficulty']}) db={r['db_id']}  ex={r['ex']}  "
          f"join_f1={jf}  ret_recall={rr}")
    print(f"    Q:    {q['question'][:140]}")
    if q.get("evidence"):
        print(f"    ev:   {q['evidence'][:140]}")
    print(f"    cands: {', '.join(r['candidate_tables']) or '(none)'}")
    print(f"    gold:  {', '.join(sorted(r['gold_tables'])) or '(none)'}")
    if missing_tables:
        print(f"    *** RETRIEVAL MISSED: {', '.join(sorted(missing_tables))}")
    print(f"    pred: {r['pred_sql'][:160]}")
    print(f"    gold: {q['SQL'][:160]}")
    if r["pred_err"]:
        print(f"    err:  {r['pred_err'][:160]}")
    shown += 1

## 15. Run BIRD's official evaluation scripts

Drop `evaluation.py` and `evaluation_ves.py` from BIRD's `evaluation/` folder into `/content/bird_eval/`. The cell builds the gold-SQL file in BIRD's expected format and invokes both scripts on the retrieval predictions.

In [ ]:
BIRD_EVAL = pathlib.Path("/content/bird_eval")
BIRD_EVAL.mkdir(exist_ok=True)

GT_PATH = BIRD_EVAL / "dev_gold.sql"
lines = []
for q in questions:
    sql = q["SQL"].replace("\n", " ").strip()
    lines.append(f"{sql}\t{q['db_id']}")
GT_PATH.write_text("\n".join(lines))
print("Wrote", GT_PATH)

ex_script  = BIRD_EVAL / "evaluation.py"
ves_script = BIRD_EVAL / "evaluation_ves.py"

if ex_script.exists():
    !python {ex_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation.py at {ex_script} and re-run.")

if ves_script.exists():
    !python {ves_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation_ves.py at {ves_script} and re-run.")

## Notes

**Component contribution:** Δ EX between this run and `baseline.ipynb` = the contribution attributable to schema linking via retrieval alone. The graph + rewriter components add on top of this in subsequent ablations.

**Retrieval-recall ceiling.** If `ret_recall` is, say, 90%, then 10% of questions are *unsolvable* under this retrieval setting because a required table never enters the prompt. The gen LLM cannot recover from those — that's the bottleneck. Tune `N_CANDIDATE_TABLES` upward if recall is low; tune downward if recall is saturated and the LLM is still distracted.

**Throughput.** Expansion (Qwen-7B BF16) is ~0.3–0.5 sec/question. Retrieval is sub-millisecond after warm cache. SQL generation (Qwen-32B 4-bit) is ~3–5 sec/question. Full BIRD dev (1534 Q): ~1.5–2.5 hr — same ballpark as baseline.

**Knobs:**
- `N_CANDIDATE_TABLES` (cell 10) — how many tables Qwen-32B sees. 5 by default.
- `RET_GEN_BATCH`, `RET_EXP_BATCH` env vars — generation / expansion batch sizes.
- Expansion strategy: 3 paraphrases + entity extraction, no sub-questions. To experiment with sub-questions, edit `EXPANSION_SYSTEM` and the few-shot example.